# RXN2 relation extraction — resume with live status
This resumes previously completed records from Drive. Use Runtime → Change runtime type → T4 GPU.


In [ ]:
!pip -q install "transformers==4.57.6" "accelerate>=1.10,<2" "huggingface_hub>=0.34,<1"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'
print(torch.cuda.get_device_name(0))
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME='Qwen/Qwen3-4B-Instruct-2507'
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,torch_dtype=torch.float16,device_map='auto',trust_remote_code=True,low_cpu_mem_usage=True).eval()
print('loaded',MODEL_NAME)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib,json,os,torch
ROOT=Path('/content/drive/MyDrive/RXN2/relation-extraction')
INPUT=ROOT/'jobs/jobs.jsonl'; OUTPUT=ROOT/'results/results.jsonl'; RETRY=ROOT/'retry/retry.jsonl'; STATUS=ROOT/'status.json'
assert INPUT.is_file(), f'Missing input: {INPUT}'
OUTPUT.parent.mkdir(parents=True,exist_ok=True); RETRY.parent.mkdir(parents=True,exist_ok=True)
SYSTEM='''Extract factual relations from one performed patent procedure. Return JSON only. Copy names and supporting_quote verbatim. Never generate SMILES, InChIKeys, reaction SMILES, identities, quantities, yields, or conditions that are not explicit. If not performed set procedure_performed=false. Never approve chemistry. Schema: {procedure_performed:boolean, materials:[{surface_text:string,role:string,explicit:boolean}], conditions:object, outcome:object, supporting_quote:string, uncertainties:[string]}'''
def now(): return datetime.now(timezone.utc).isoformat()
def digest(j): return hashlib.sha256((j['evidence_span_id']+'\n'+j['evidence_text']).encode()).hexdigest()
def atomic_json(path,value):
    tmp=path.with_suffix('.json.partial'); tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding='utf-8'); tmp.replace(path)
def extract(text):
    x=tokenizer(SYSTEM+'\n\nPROCEDURE:\n'+text,return_tensors='pt',truncation=True,max_length=12000).to(model.device)
    with torch.inference_mode(): y=model.generate(**x,max_new_tokens=768,do_sample=False,temperature=0.0,pad_token_id=tokenizer.eos_token_id)
    raw=tokenizer.decode(y[0][x['input_ids'].shape[1]:],skip_special_tokens=True); a,b=raw.find('{'),raw.rfind('}')+1
    if a<0 or b<=a: raise ValueError('model did not return JSON')
    return json.loads(raw[a:b])


In [ ]:
# Run this cell to resume. It never repeats a successful hash.
jobs=[json.loads(line) for line in INPUT.read_text(encoding='utf-8').splitlines() if line.strip()]
done=set()
if OUTPUT.exists():
    for line in OUTPUT.read_text(encoding='utf-8').splitlines():
        if line.strip(): done.add(json.loads(line)['input_sha256'])
started=now(); errors=0
atomic_json(STATUS,{'state':'running','total':len(jobs),'completed':len(done),'errors':errors,'started_at':started,'updated_at':started})
with OUTPUT.open('a',encoding='utf-8') as out, RETRY.open('a',encoding='utf-8') as retry:
    for job in jobs:
        h=digest(job)
        if h in done: continue
        atomic_json(STATUS,{'state':'running','total':len(jobs),'completed':len(done),'errors':errors,'current_evidence_span_id':job['evidence_span_id'],'started_at':started,'updated_at':now()})
        try:
            rec={'input_sha256':h,'evidence_span_id':job['evidence_span_id'],'publication_number':job.get('publication_number'),'candidate':extract(job['evidence_text']),'model':MODEL_NAME,'created_at':now()}
            out.write(json.dumps(rec,ensure_ascii=False)+'\n'); out.flush(); os.fsync(out.fileno()); done.add(h)
        except Exception as exc:
            errors+=1; retry.write(json.dumps({'input_sha256':h,'evidence_span_id':job['evidence_span_id'],'error':str(exc),'created_at':now()})+'\n'); retry.flush(); os.fsync(retry.fileno())
        atomic_json(STATUS,{'state':'running','total':len(jobs),'completed':len(done),'errors':errors,'current_evidence_span_id':job['evidence_span_id'],'started_at':started,'updated_at':now()})
        print(f'progress {len(done)}/{len(jobs)} | errors {errors} | latest {job["evidence_span_id"]}',flush=True)
atomic_json(STATUS,{'state':'completed','total':len(jobs),'completed':len(done),'errors':errors,'started_at':started,'updated_at':now()})
print('run complete:',len(done),'of',len(jobs),'successful; errors:',errors)
